<img src = "imgs/tonic-logo-black-bg.webp" width="10%">

In [1]:
from tonic.transforms import ToFrame
from torch.utils.data import DataLoader
from tonic.datasets.nmnist import NMNIST
import matplotlib.pyplot as plt
import numpy as np
n_time_steps = 10
batch_size = 16
root_dir = '/home/mazdak/projects/univ-lille/datasets'    

to_raster = ToFrame(sensor_size=NMNIST.sensor_size, n_time_bins=n_time_steps)

train_dataset = NMNIST(save_to=root_dir, train=True, transform=to_raster)
test_dataset = NMNIST(save_to=root_dir, train=False, transform=to_raster)

train_dataloader = DataLoader( train_dataset, batch_size=batch_size, drop_last=True, shuffle=True)
test_dataloader = DataLoader(test_dataset,batch_size=batch_size, drop_last=True,shuffle=True)


/home/mazdak/projects/univ-lille/datasets/NMNIST/train.zip
/home/mazdak/projects/univ-lille/datasets/NMNIST/test.zip


In [2]:
d, t = iter(test_dataloader).__next__()
d.shape

torch.Size([16, 10, 2, 34, 34])

# Design Model/ Train/ Test
<img src = "imgs/sinabs-logo-lowercase-whitebg.webp" width="20%">

In [3]:
from torch import nn
import torch
import sinabs.layers as sl

model = nn.Sequential(
    # sl.FlattenTime(),  # use it if you don't want to reshape the data by yourself
    nn.Conv2d( in_channels=2, out_channels=8, kernel_size=(3, 3), padding=(1, 1), bias=False),
    nn.ReLU(),
    nn.AvgPool2d(2, 2),
    # [8, 17, 17] -> [16, 8, 8]
    nn.Conv2d( in_channels=8, out_channels=16, kernel_size=(3, 3), padding=(1, 1), bias=False ),
    nn.ReLU(),
    nn.AvgPool2d(2, 2),
    # [16 * 8 * 8] -> [16, 4, 4]
    nn.Conv2d( in_channels=16, out_channels=16, kernel_size=(3, 3), padding=(1, 1), stride=(2, 2), bias=False),
    nn.ReLU(),
    # [16 * 4 * 4] -> [10]
    nn.Flatten(),
    nn.Linear(16 * 4 * 4, 10, bias=False),
    nn.ReLU(),
    # sl.UnflattenTime(batch_size=batch_size) ## use it if you don't want to reshape the data by yourself
)

# init the model weights
for layer in model.modules():
    if isinstance(layer, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_normal_(layer.weight.data)

In [4]:
import torch
# model(d.to(dtype=torch.float)).shape # using FlattenTime
model(d.reshape(-1, 2, 34, 34).to(torch.float)).shape

torch.Size([160, 10])

### Sinbas Model

In [5]:
from sinabs.from_torch import from_model
sinabs_model = from_model(model, batch_size=batch_size)#.to(device)

`NOTE:`

The ReLU functions, are replaced with IAFSqueeze which is time-step-aware

In [6]:
sinabs_model

Network(
  (spiking_model): Sequential(
    (0): Conv2d(2, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): IAFSqueeze(spike_threshold=1.0, min_v_mem=-1.0, batch_size=16, num_timesteps=-1)
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): IAFSqueeze(spike_threshold=1.0, min_v_mem=-1.0, batch_size=16, num_timesteps=-1)
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (6): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (7): IAFSqueeze(spike_threshold=1.0, min_v_mem=-1.0, batch_size=16, num_timesteps=-1)
    (8): Flatten(start_dim=1, end_dim=-1)
    (9): Linear(in_features=256, out_features=10, bias=False)
    (10): IAFSqueeze(spike_threshold=1.0, min_v_mem=-1.0, batch_size=16, num_timesteps=-1)
  )
  (analog_model): Sequential(
    (0): Conv2d(2, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1):

### Train Model

In [7]:
from torch.optim import SGD
from torch.nn import CrossEntropyLoss
epochs = 30
lr = 1e-2
optimizer = SGD(params=sinabs_model.parameters(), lr=lr)
criterion = CrossEntropyLoss()


In [ ]:
from tqdm.notebook import tqdm
for epoch in range(1):
    pbar = tqdm(train_dataloader)
    for data, target in pbar:
        optimizer.zero_grad()
        sinabs_model.reset_states()

        
        data = data.reshape(-1, 2, 34, 34).to(dtype=torch.float) # merging time and batch dimensions  #to(dtype=torch.float)
        out = sinabs_model.spiking_model(data)#.to(device))
        out = out.reshape(batch_size, n_time_steps, -1)  # restoring original dimensions

        # the output of the network is summed over the time steps 
        out = out.sum(1)
        loss = criterion(out, target)#.to(device))
        loss.backward()
        optimizer.step()

        pbar.set_postfix(loss=loss.item())

In [ ]:
# torch.save(sinabs_model, "NMNIST_sinabs_model.pth")

### Test Model

In [ ]:
accs = []

pbar = tqdm(test_dataloader)
for data, target in pbar:
    sinabs_model.reset_states()

    data = data.reshape(-1, 2, 34, 34).to(dtype=torch.float) # merging time and batch dimensions #to(dtype=torch.float)
    out = sinabs_model.spiking_model(data)
    out = out.reshape(batch_size, n_time_steps, -1)  # restoring original dimensions

    out = out.sum(1)
    predicted = torch.max(out, axis=1)[1]
    acc = (predicted == target).sum().cpu().numpy() / batch_size
    accs.append(acc)

print(sum(accs) / len(accs))

# Deploy On SPECK

### DynapCNN model - `no` DVS

In [8]:
from sinabs.backend.dynapcnn import DynapcnnNetwork
dynapcnn_model = DynapcnnNetwork(sinabs_model, input_shape= (2, 34, 34), dvs_input=False)

`NOTE:`
- The Pooling (avg) are replaced with SumPool2d
- The `Linear layer` => `Conv2d`

In [9]:
dynapcnn_model

DynapcnnNetwork(
  (sequence): Sequential(
    (0): DynapcnnLayer(
      (conv_layer): Conv2d(2, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (spk_layer): IAFSqueeze(spike_threshold=347.0, min_v_mem=-347.0, batch_size=16, num_timesteps=-1)
      (pool_layer): SumPool2d(norm_type=1, kernel_size=(2, 2), stride=None, ceil_mode=False)
    )
    (1): DynapcnnLayer(
      (conv_layer): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (spk_layer): IAFSqueeze(spike_threshold=1353.0, min_v_mem=-1353.0, batch_size=16, num_timesteps=-1)
      (pool_layer): SumPool2d(norm_type=1, kernel_size=(2, 2), stride=None, ceil_mode=False)
    )
    (2): DynapcnnLayer(
      (conv_layer): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (spk_layer): IAFSqueeze(spike_threshold=1765.0, min_v_mem=-1765.0, batch_size=16, num_timesteps=-1)
    )
    (3): DynapcnnLayer(
      (conv_layer): Conv2d(16, 10, kernel_size=(4,

In [ ]:
devkit_name = "speck2fdevkit"
dynapcnn_model.to(device=devkit_name, chip_layers_ordering="auto")
print(f"The SNN is deployed on the core: {dynapcnn_model.chip_layers_ordering}")

### Test on HW

In [ ]:
from sinabs.backend.dynapcnn.chip_factory import ChipFactory
factory = ChipFactory(devkit_name)

In [ ]:
inferece_p_bar = tqdm(test_dataloader)
test_samples = 0
correct_samples = 0
first_layer_idx = dynapcnn_model.chip_layers_ordering[0]
for data_batch, label_batch in inferece_p_bar:
    for data, label in zip(data_batch, label_batch):
        print(data.shape)
        speck_event = factory.raster_to_events(data, layer=first_layer_idx)
        output_events = dynapcnn_model(speck_event)
        neuron_index = [each.feature for each in output_events]
        freq = np.unique( neuron_index, return_counts=True)
        prediction = freq[0][freq[1].argmax()]

        inferece_p_bar.set_description(f"label: {label}, prediction: {prediction}， output spikes num: {len(output_events)}") 

        if prediction == label:
            correct_samples += 1

        test_samples += 1
        
print(f"On chip inference accuracy: {correct_samples / test_samples}") 

In [ ]:
correct_samples/ test_samples

### NEXT: DynapCNN model - DVS Live!